In [53]:
# Analisis explotario de SIMCE
import pandas as pd
import numpy as np


rutasimce = r"C:\Users\Manuel\Escritorio\DataScience\Diplomado Ciencia e Ingenieria de Datos\1. - Taller de Proyecto\Proyecto Diplomado - Educacion\Datos\1-. Resultados SIMCE\2M\Archivos CSV (Planos)\simce2m2024_rbd_preliminar.csv"
rutaidps = r"C:\Users\Manuel\Escritorio\DataScience\Diplomado Ciencia e Ingenieria de Datos\1. - Taller de Proyecto\Proyecto Diplomado - Educacion\Datos\2-. IDPS\2M\Archivos CSV (Planos)\idps2M2024_rbd_preliminar.csv"

simce = pd.read_csv(rutasimce, sep=";", encoding= "cp1252")
idps = pd.read_csv(rutaidps, sep=";", encoding= "cp1252")

In [54]:
print(f'Simce: {simce.shape}')
print(f'idps: {idps.shape}\n')

Simce: (3000, 42)
idps: (12000, 22)



In [ ]:
columnas_sel = [
    "rbd", "nom_rbd", "cod_reg_rbd", "nom_reg_rbd", "cod_com_rbd", "nom_com_rbd",
    "cod_depe2", "cod_grupo", "cod_rural_rbd", "nalu_mate2m_rbd", "prom_mate2m_rbd",
    "noaplica"
]
# marca_mate2m_rbd
base_simce = simce[columnas_sel].copy()

base_simce.head()

,rbd,nom_rbd,cod_reg_rbd,nom_reg_rbd,cod_com_rbd,nom_com_rbd,cod_depe2,cod_grupo,cod_rural_rbd,nalu_mate2m_rbd,prom_mate2m_rbd,noaplica,marca_mate2m_rbd
0,7826,COLEGIO SAN MIGUEL,10,DE LOS LAGOS,10102,CALBUCO,2,3.0,1,106,240.0,0,NaN
1,6830,COLEGIO NUESTRA SENORA DEL CARMEN,14,DE LOS RÍOS,14101,VALDIVIA,2,3.0,1,78,240.0,0,NaN
2,40126,COLEGIO RAUL SILVA HENRIQUEZ,4,DE COQUIMBO,4301,OVALLE,1,1.0,1,67,233.0,0,NaN
3,18237,COLEGIO AMANECER SAN CARLOS,8,DEL BIOBÍO,8108,SAN PEDRO DE LA PAZ,2,4.0,1,84,267.0,0,NaN
4,14324,COLEGIO WILLIAM JAMES,5,DE VALPARAÍSO,5109,VIÑA DEL MAR,3,5.0,1,11,261.0,0,NaN


In [56]:
map_gse   = {1: "Bajo", 2: "Medio bajo", 3: "Medio", 4: "Medio alto", 5: "Alto"}
map_depe2 = {1: "Municipal", 2: "Part. subvencionado", 3: "Part. pagado", 4: "SLEP"}
map_rural = {1: "Urbano", 2: "Rural"}

base_simce["gse"] = base_simce["cod_grupo"].map(map_gse)
base_simce["depe"] = base_simce["cod_depe2"].map(map_depe2)
base_simce["zona"] = base_simce["cod_rural_rbd"].map(map_rural)

print(base_simce.shape)
base_simce.head()

(3000, 15)


,rbd,nom_rbd,cod_reg_rbd,nom_reg_rbd,cod_com_rbd,nom_com_rbd,cod_depe2,cod_grupo,cod_rural_rbd,nalu_mate2m_rbd,prom_mate2m_rbd,noaplica,gse,depe,zona
0,7826,COLEGIO SAN MIGUEL,10,DE LOS LAGOS,10102,CALBUCO,2,3.0,1,106,240.0,0,Medio,Part. subvencionado,Urbano
1,6830,COLEGIO NUESTRA SENORA DEL CARMEN,14,DE LOS RÍOS,14101,VALDIVIA,2,3.0,1,78,240.0,0,Medio,Part. subvencionado,Urbano
2,40126,COLEGIO RAUL SILVA HENRIQUEZ,4,DE COQUIMBO,4301,OVALLE,1,1.0,1,67,233.0,0,Bajo,Municipal,Urbano
3,18237,COLEGIO AMANECER SAN CARLOS,8,DEL BIOBÍO,8108,SAN PEDRO DE LA PAZ,2,4.0,1,84,267.0,0,Medio alto,Part. subvencionado,Urbano
4,14324,COLEGIO WILLIAM JAMES,5,DE VALPARAÍSO,5109,VIÑA DEL MAR,3,5.0,1,11,261.0,0,Alto,Part. pagado,Urbano


In [57]:
na = base_simce.isna().sum()
na[na > 0].sort_values(ascending=False)

prom_mate2m_rbd    6
cod_grupo          5
gse                5
dtype: int64

In [58]:
base_simce = base_simce.dropna(subset=["cod_grupo"]).copy()
base_simce = base_simce.dropna(subset=["prom_mate2m_rbd"]).copy()
base_simce = base_simce.dropna(subset=["gse"]).copy()

base_simce.shape

(2994, 15)

In [59]:
mask = base_simce["gse"].isin(['Bajo', 'Medio bajo'])
base_simce[mask].pivot_table(
    index="gse",
    values="rbd",
    aggfunc="nunique",
    margins=True,
    margins_name="Total"
)

,rbd
gse,
Bajo,693
Medio bajo,867
Total,1560


In [60]:
sub_bs = base_simce[mask].copy()

c1, c2 = 252, 319
cortes    = [-np.inf, c1, c2, np.inf]
etiquetas = ["Insuficiente", "Elemental", "Adecuado"]

sub_bs["nivel"] = pd.cut(sub_bs["prom_mate2m_rbd"], bins=cortes,
                      labels=etiquetas, right=False)

sub_bs.groupby("nivel", observed=False)["rbd"].count()

nivel
Insuficiente    1186
Elemental        357
Adecuado          17
Name: rbd, dtype: int64

In [67]:
sub_bs["target_bin"] = np.where(sub_bs["nivel"] == "Insuficiente", "Insuficiente", "Elemental o adecuado")
sub_bs["target_bin"].value_counts()

target_bin
Insuficiente            1186
Elemental o adecuado     374
Name: count, dtype: int64

In [68]:
sub_bs.head()

,rbd,nom_rbd,cod_reg_rbd,nom_reg_rbd,cod_com_rbd,nom_com_rbd,cod_depe2,cod_grupo,cod_rural_rbd,nalu_mate2m_rbd,prom_mate2m_rbd,noaplica,gse,depe,zona,nivel,target_bin
2,40126,COLEGIO RAUL SILVA HENRIQUEZ,4,DE COQUIMBO,4301,OVALLE,1,1.0,1,67,233.0,0,Bajo,Municipal,Urbano,Insuficiente,Insuficiente
5,6853,LICEO POLITECNICO PESQUERO,14,DE LOS RÍOS,14106,MARIQUINA,1,1.0,2,16,213.0,0,Bajo,Municipal,Rural,Insuficiente,Insuficiente
6,26372,COLEGIO SAN LUCAS DE LO ESPEJO,13,METROPOLITANA DE SANTIAGO,13116,LO ESPEJO,2,1.0,1,63,259.0,0,Bajo,Part. subvencionado,Urbano,Elemental,Elemental o adecuado
9,22436,THE MISSION COLLEGE,10,DE LOS LAGOS,10301,OSORNO,2,1.0,1,108,241.0,0,Bajo,Part. subvencionado,Urbano,Insuficiente,Insuficiente
10,12117,COMPLEJO EDUCACIONAL J. MIGUEL CARRERA,13,METROPOLITANA DE SANTIAGO,13125,QUILICURA,1,1.0,1,129,222.0,0,Bajo,Municipal,Urbano,Insuficiente,Insuficiente


In [62]:
na_idps = idps.isna().sum()
na_idps[na_idps > 0].sort_values(ascending=False)

dif          1115
sigdif       1115
sigdifgru     105
difgru        105
prom           45
cod_grupo      20
dtype: int64

In [63]:
map_ind = {
    "AM": "autoestima_motiv",
    "CC": "convivencia",
    "PF": "participacion",
    "HV": "vida_saludable",
}

idps["prom"] = pd.to_numeric(idps["prom"], errors="coerce")

idps_ancho = (idps
             .pivot_table(index="rbd", columns="ind", values="prom")
             .rename(columns=map_ind)
             .reset_index())
idps_ancho.columns.name = None

print("idps_wide:", idps_ancho.shape)
idps_ancho.head(3)

idps_wide: (2992, 5)


,rbd,autoestima_motiv,convivencia,vida_saludable,participacion
0,1,74.0,75.0,74.0,79.0
1,4,74.0,74.0,70.0,75.0
2,5,71.0,73.0,71.0,75.0


In [64]:
base = sub_bs.merge(idps_ancho, on="rbd", how="left")

na_base = base.isna().sum()
print(na_base[na_base > 0].sort_values())
print()
print(base.shape)

convivencia    7
dtype: int64

(1560, 21)


In [70]:
base.head()

,rbd,nom_rbd,cod_reg_rbd,nom_reg_rbd,cod_com_rbd,nom_com_rbd,cod_depe2,cod_grupo,cod_rural_rbd,nalu_mate2m_rbd,...,noaplica,gse,depe,zona,nivel,target_bin,autoestima_motiv,convivencia,vida_saludable,participacion
0,40126,COLEGIO RAUL SILVA HENRIQUEZ,4,DE COQUIMBO,4301,OVALLE,1,1.0,1,67,...,0,Bajo,Municipal,Urbano,Insuficiente,Insuficiente,74.0,75.0,73.0,79.0
1,6853,LICEO POLITECNICO PESQUERO,14,DE LOS RÍOS,14106,MARIQUINA,1,1.0,2,16,...,0,Bajo,Municipal,Rural,Insuficiente,Insuficiente,72.0,71.0,73.0,73.0
2,26372,COLEGIO SAN LUCAS DE LO ESPEJO,13,METROPOLITANA DE SANTIAGO,13116,LO ESPEJO,2,1.0,1,63,...,0,Bajo,Part. subvencionado,Urbano,Elemental,Elemental o adecuado,76.0,79.0,76.0,81.0
3,22436,THE MISSION COLLEGE,10,DE LOS LAGOS,10301,OSORNO,2,1.0,1,108,...,0,Bajo,Part. subvencionado,Urbano,Insuficiente,Insuficiente,72.0,73.0,72.0,74.0
4,12117,COMPLEJO EDUCACIONAL J. MIGUEL CARRERA,13,METROPOLITANA DE SANTIAGO,13125,QUILICURA,1,1.0,1,129,...,0,Bajo,Municipal,Urbano,Insuficiente,Insuficiente,74.0,73.0,71.0,78.0
